# CSV Loading Into Pandas DataFrames

This milestone focuses on loading CSV data safely, inspecting structure immediately, and catching common loading issues early.

## Learning goals
- Understand CSV rows, columns, headers, and delimiters
- Load CSV files using `pandas.read_csv()`
- Verify DataFrame structure after loading
- Detect basic loading issues before analysis

## CSV Concept Refresher

A CSV file is a plain-text table.
- Each line usually represents one row.
- Values are separated by a delimiter (most often a comma).
- The first line usually contains headers (column names).

If delimiter or header handling is wrong, your DataFrame structure can be incorrect from the beginning.

## 1. Setup Environment and Import Pandas

We import Pandas and set display options to keep previews readable.

In [ ]:
from pathlib import Path
from io import StringIO

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 2. Define File Paths and Validate File Availability

Before loading, confirm the file path is correct. This avoids silent path mistakes.

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
csv_path = project_root / "data" / "raw" / "employee_survey_2026_Q1.csv"

print(f"Project root: {project_root}")
print(f"CSV path: {csv_path}")
print(f"File exists: {csv_path.exists()}")

## 3. Load CSV with Default `pd.read_csv()`

Default loading treats the first row as headers (`header=0` behavior).

In [ ]:
df_survey = pd.read_csv(csv_path)

print(type(df_survey))
print(df_survey.shape)

## 4. Inspect Rows, Columns, and Shape

Immediate inspection prevents hidden loading issues.

In [ ]:
df_survey.head()

In [ ]:
df_survey.tail(3)

In [ ]:
print("Shape (rows, columns):", df_survey.shape)
print("Row count using len(df):", len(df_survey))
print("Column count:", len(df_survey.columns))

## 5. Verify Header Interpretation

Check whether Pandas correctly interpreted the first row as column headers.

In [ ]:
print("Detected columns:")
print(list(df_survey.columns))

In [ ]:
# Demonstration only: forcing header=None makes the original header row become data.
df_header_none = pd.read_csv(csv_path, header=None)
df_header_none.head(3)

In [ ]:
# Demonstration only: assign manual names if a file has no header row.
manual_names = [
    "employee_id",
    "department",
    "survey_date",
    "satisfaction_score",
    "work_life_balance",
    "management_support",
    "career_growth",
    "team_collaboration",
    "response_text",
]

df_manual_names = pd.read_csv(csv_path, header=0, names=manual_names)
print(list(df_manual_names.columns))

## 6. Check Column Names and Basic Structure

Validate column labels, index behavior, non-null counts, and inferred data types.

In [ ]:
print("Column names:")
print(list(df_survey.columns))

print("\nIndex preview:")
print(df_survey.index[:5])

print("\nDataFrame info:")
df_survey.info()

## 7. Detect Common CSV Loading Issues

These checks help you notice wrong delimiter usage, unnamed columns, or column-count mismatches.

In [ ]:
expected_columns = [
    "employee_id",
    "department",
    "survey_date",
    "satisfaction_score",
    "work_life_balance",
    "management_support",
    "career_growth",
    "team_collaboration",
    "response_text",
]

# Wrong delimiter example using in-memory CSV text.
sample_csv = (
    "employee_id,department,satisfaction_score\n"
    "1001,Engineering,7\n"
    "1002,Marketing,9\n"
)

df_wrong_sep = pd.read_csv(StringIO(sample_csv), sep=";")
print("Wrong delimiter shape:", df_wrong_sep.shape)
print("Wrong delimiter columns:", list(df_wrong_sep.columns))

# Quick structural checks on the real DataFrame.
unnamed_columns = [col for col in df_survey.columns if str(col).startswith("Unnamed")]
missing_columns = [col for col in expected_columns if col not in df_survey.columns]
extra_columns = [col for col in df_survey.columns if col not in expected_columns]

print("\nUnnamed columns:", unnamed_columns)
print("Missing expected columns:", missing_columns)
print("Unexpected extra columns:", extra_columns)
print("Expected column count:", len(expected_columns))
print("Actual column count:", len(df_survey.columns))

## 8. Apply Defensive Loading Checks

Use simple assertions to fail early when the loaded structure is not what you expect.

In [ ]:
assert all(col in df_survey.columns for col in expected_columns), "Missing expected columns in loaded DataFrame"
assert len(df_survey) >= 25, "Row count seems too low for this dataset"
assert len(df_survey.columns) == len(expected_columns), "Column count mismatch detected"

print("Defensive checks passed: structure looks valid for further analysis.")

## 9. Bonus: Read CSV with Alternate Delimiters and Encodings

Some CSV files are not comma-separated or may use specific encodings. This demonstrates how to handle those formats.

In [ ]:
semicolon_csv = (
    "employee_id;department;satisfaction_score\n"
    "2001;Engineering;8\n"
    "2002;Sales;5\n"
)

df_semicolon = pd.read_csv(StringIO(semicolon_csv), sep=";")
print("Semicolon-separated DataFrame:")
print(df_semicolon)

# Encoding/quote examples (shown as patterns for non-standard files):
# pd.read_csv("file.csv", encoding="utf-8")
# pd.read_csv("file.csv", encoding="latin-1")
# pd.read_csv("file.csv", quotechar='"')

## Final Takeaways

- Load CSV files with `pd.read_csv()` only after confirming the file path.
- Immediately inspect `head()`, `columns`, `shape`, and `info()`.
- Verify expected columns and row counts before any analysis.
- Catch loading issues early (wrong delimiter, header misread, unexpected columns).